# Codenames Spymaster RL Smoke Test

This notebook exercises the current end-to-end pipeline:

1. Load the smoke-test config
2. Build the embedding store and clue vocabulary
3. Reset the goal-conditioned spymaster environment and inspect the state
4. Run the greedy spymaster baseline for one episode
5. Generate demonstration data for behavioral cloning
6. Run the SAC + HER training smoke test and print evaluation metrics


In [1]:
import json
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.append(str(PROJECT_ROOT))

from src.agents.bc_pretrain import generate_demonstrations
from src.baselines.greedy_spymaster import GreedySpymaster
from src.evaluation.evaluate_agent import evaluate_agent
from src.env.visualization import plot_board
from src.training.train_sac_her import (
    build_embedding_store,
    build_env_factory,
    load_config,
    run_training_pipeline,
)


ModuleNotFoundError: No module named 'yaml'

In [ ]:
config = load_config(PROJECT_ROOT / "configs" / "smoke_test.yaml")
config


In [ ]:
embedding_store = build_embedding_store(config)
env_factory = build_env_factory(config, embedding_store)
env = env_factory()
obs, info = env.reset()

print("Board words:", info["board_words"])
print("Role counts:", info["role_counts"])
print("Observation shape:", obs["observation"].shape)
print("Achieved goal shape:", obs["achieved_goal"].shape)
print("Desired goal shape:", obs["desired_goal"].shape)
print("Number of clue candidates:", len(embedding_store.clue_words))

plot_board(env.board, reveal_roles=True, title="Spymaster View")


In [ ]:
greedy_agent = GreedySpymaster()
greedy_metrics = evaluate_agent(greedy_agent, env_factory=env_factory, episodes=1)
greedy_metrics


In [ ]:
demos = generate_demonstrations(
    env_factory=env_factory,
    num_episodes=config["bc"]["demo_episodes"],
    max_steps_per_episode=config["env"]["max_turns"],
)
print("Collected demonstration transitions:", len(demos))
print("Sample action vector shape:", demos[0].action.shape)
print("Sample reward:", demos[0].reward)
print("Sample clue:", demos[0].info["clue"])


In [ ]:
results = run_training_pipeline(config)
print(json.dumps(results, indent=2))
